# PPO Environment for Synthetic Temporal Early-Stage Ransomware Detection

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Notebook 04:** Synthetic Temporal Proxy Gymnasium Environment Specification and Validation  

**Purpose:**  
Design, implement, test, and validate a clean Gymnasium-compatible environment (`RansomwareDetectionEnv`) that serves as the sequential decision-making foundation for future PPO reinforcement learning in `notebooks/05_train_ppo.ipynb`.

> **IMPORTANT SCOPE NOTE:**  
> PPO agent training is **NOT** performed in this notebook. This phase focuses exclusively on environment mechanics, observation/action space formulation, progressive feature reveal, reward calibration, and empirical validation.


## Research Caveat: Synthetic Temporal Proxy Framing

> **CRITICAL SCIENTIFIC INVARIANT:**  
> The current dataset (`cleaned_dataset.csv`) is a static dynamic-analysis summary dataset. It does **NOT** contain real microsecond timestamps, sequential API call streams, or dynamic memory snapshots over physical execution time.  
>  
> **Therefore:**  
> - This environment implements a **synthetic temporal proxy** using progressive feature revelation.  
> - Step indices ($t = 0, 1, 2, 3, 4$) represent **synthetic observation stages**, **NOT** real execution seconds or milliseconds.  
> - No live malware is executed.  
> - Genuine real-time early detection will require actual timestamped memory/API telemetry in future work.


In [ ]:
import json
import sys
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

print("Imports loaded successfully. Gymnasium version:", gym.__version__)


In [ ]:
# Resolve project root dynamically using pathlib
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_dir = project_root / "data" / "processed"
figures_dir = project_root / "results" / "figures"
reports_dir = project_root / "results" / "reports"

figures_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Data dir     : {data_dir}")
print(f"Figures dir  : {figures_dir}")
print(f"Reports dir  : {reports_dir}")


In [ ]:
cleaned_data_path = data_dir / "cleaned_dataset.csv"
df = pd.read_csv(cleaned_data_path)

print(f"Cleaned Dataset Shape: {df.shape}")
print("Target Column        : 'family'")
print(f"Missing Values       : {df.isnull().sum().sum()}")
print(f"Duplicate Rows       : {df.duplicated().sum()}")
print("Class Distribution   :")
print(df["family"].value_counts())

assert df.shape == (1750, 47), "Dataset shape mismatch!"
assert df.isnull().sum().sum() == 0, "Found unexpected nulls!"
assert df.duplicated().sum() == 0, "Found unexpected duplicates!"


In [ ]:
feature_list_path = data_dir / "early_stage_nonredundant_feature_list.json"
with open(feature_list_path, "r", encoding="utf-8") as f:
    feature_names = json.load(f)

print(f"Loaded {len(feature_names)} non-redundant early-stage features:")
for i, feat in enumerate(feature_names, start=1):
    print(f"  {i:2d}. {feat}")

# Assert all 16 features exist in dataset
for feat in feature_names:
    assert feat in df.columns, f"Feature {feat} missing from dataset!"


In [ ]:
split_path = data_dir / "split_indices.json"
with open(split_path, "r", encoding="utf-8") as f:
    split_info = json.load(f)

train_indices = split_info["train_indices"]
test_indices = split_info["test_indices"]

print(f"Training Samples : {len(train_indices)} ({len(train_indices)/len(df):.1%})")
print(f"Testing Samples  : {len(test_indices)} ({len(test_indices)/len(df):.1%})")

train_classes = df.iloc[train_indices]["family"].value_counts().to_dict()
test_classes = df.iloc[test_indices]["family"].value_counts().to_dict()

print(f"Train Class Dist : {train_classes}")
print(f"Test Class Dist  : {test_classes}")

assert len(train_indices) == 1400, "Train size must be exactly 1400"
assert len(test_indices) == 350, "Test size must be exactly 350"


In [ ]:
from src.ppo_environment import fit_and_save_normalization

norm_config_path = data_dir / "ppo_normalization.json"
norm_config = fit_and_save_normalization(
    data_path=cleaned_data_path,
    split_indices_path=split_path,
    feature_list_path=feature_list_path,
    output_path=norm_config_path,
)

print(f"StandardScaler fit strictly on {norm_config['n_samples_fit']} training samples.")
print(f"Normalization configuration saved to: {norm_config_path}")


In [ ]:
train_df = df.iloc[train_indices][feature_names]
test_df = df.iloc[test_indices][feature_names]

scaler_mean = np.array(norm_config["mean"])
scaler_scale = np.array(norm_config["scale"])

train_norm = (train_df.to_numpy() - scaler_mean) / scaler_scale
test_norm = (test_df.to_numpy() - scaler_mean) / scaler_scale

print("Training Normalized Means (Target ~0.0):")
print(np.round(train_norm.mean(axis=0), 4))

print("Training Normalized Std Devs (Target ~1.0):")
print(np.round(train_norm.std(axis=0), 4))

print("Test Normalized Means (Applied without re-fitting):")
print(np.round(test_norm.mean(axis=0), 4))


In [ ]:
TEMPORAL_CONFIG = {
    "sequence_length": 5,
    "reveal_schedule": [0.2, 0.4, 0.6, 0.8, 1.0],
    "mask_value": 0.0
}

print("Synthetic Temporal Proxy Configuration:")
for k, v in TEMPORAL_CONFIG.items():
    print(f"  {k:20s}: {v}")
print("Note: These 5 observation stages represent synthetic progressive information reveal, NOT real execution seconds.")


In [ ]:
from src.ppo_environment import generate_temporal_sequence

# Test generator with a sample training vector
sample_vector = train_norm[0]
synthetic_seq = generate_temporal_sequence(sample_vector, TEMPORAL_CONFIG)

print(f"Generated synthetic sequence shape: {synthetic_seq.shape} (Expected: 5 x 16)")
assert synthetic_seq.shape == (5, 16), "Sequence shape mismatch!"


In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(
    synthetic_seq,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    xticklabels=feature_names,
    yticklabels=[f"Step {t} ({int(TEMPORAL_CONFIG['reveal_schedule'][t]*100)}% reveal)" for t in range(5)],
    cbar_kws={"label": "Normalized Feature Value"},
    linewidths=0.5,
)
plt.title("Synthetic Temporal Proxy Observation Progression (1 Episode)\n[NOTE: Synthetic Observation Steps — NOT Real Execution Timeline]", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("16 Non-Redundant Early-Stage Features", fontsize=10, labelpad=8)
plt.ylabel("Observation Step", fontsize=10, labelpad=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(figures_dir / "ppo_synthetic_episode.png", dpi=300)
plt.show()


In [ ]:
from src.ppo_environment import (
    ACTION_CONTINUE,
    ACTION_SUSPICIOUS,
    ACTION_ALERT,
    ACTION_NAMES,
    DEFAULT_REWARD_CONFIG,
)

print("Discrete Action Definitions:")
for act_id, act_name in ACTION_NAMES.items():
    print(f"  Action {act_id}: {act_name}")

print("Centralized Reward Configuration (Research Hyperparameters):")
for r_key, r_val in DEFAULT_REWARD_CONFIG.items():
    print(f"  {r_key:35s}: {r_val}")


In [ ]:
from src.ppo_environment import RansomwareDetectionEnv

env = RansomwareDetectionEnv(
    df=df,
    feature_names=feature_names,
    split_type="train",
    split_indices_path=split_path,
    norm_config_path=norm_config_path,
    temporal_config=TEMPORAL_CONFIG,
    reward_config=DEFAULT_REWARD_CONFIG,
    seed=42,
)

print("RansomwareDetectionEnv successfully instantiated.")


In [ ]:
print(f"Observation Space : {env.observation_space}")
print(f"Observation Shape : {env.observation_space.shape}")
print(f"Observation Dtype : {env.observation_space.dtype}")

obs, info = env.reset(seed=42)
assert env.observation_space.contains(obs), "Observation space containment failed!"


In [ ]:
print(f"Action Space : {env.action_space}")
print(f"Number of Actions : {env.action_space.n}")

for a in [0, 1, 2]:
    assert env.action_space.contains(a), f"Action space missing action {a}!"
assert not env.action_space.contains(3), "Action space should only have 3 discrete actions!"


In [ ]:
obs, info = env.reset(seed=42)

print("Reset Test Results:")
print(f"  Observation Shape : {obs.shape}")
print(f"  Sample Info       : {info}")
print(f"  Observation Sample: {np.round(obs[:6], 4)}...")

# Verify true label is NOT inside observation array
assert len(obs) == 16, "Observation vector must have length 16"
assert not isinstance(obs[0], str), "Observation must be numeric, not string label!"


In [ ]:
obs, info = env.reset(seed=100)
actions_to_take = [ACTION_CONTINUE, ACTION_CONTINUE, ACTION_SUSPICIOUS, ACTION_CONTINUE, ACTION_ALERT]

step_records = []
for a in actions_to_take:
    obs, reward, terminated, truncated, info = env.step(a)
    step_records.append({
        "step": info["step"],
        "action_taken": ACTION_NAMES[a],
        "reward": reward,
        "terminated": terminated,
        "terminal_reason": info["terminal_reason"],
    })
    if terminated:
        break

step_df = pd.DataFrame(step_records)
print("Step Test Sequence Execution Table:")
display(step_df)


In [ ]:
smoke_env = RansomwareDetectionEnv(split_type="train", seed=42)
n_episodes = 100

records = []
for ep in range(n_episodes):
    obs, info = smoke_env.reset()
    total_reward = 0.0
    steps_taken = 0
    terminated = False

    while not terminated:
        a = smoke_env.action_space.sample()
        obs, reward, terminated, truncated, info = smoke_env.step(a)
        total_reward += reward
        steps_taken += 1

    records.append({
        "episode": ep + 1,
        "sample_idx": info["sample_idx"],
        "true_label": info["true_label"],
        "episode_length": steps_taken,
        "total_reward": round(total_reward, 4),
        "terminal_reason": info["terminal_reason"],
    })

stats_df = pd.DataFrame(records)
print("100-Episode Random Policy Smoke Test Completed:")
print(f"  Mean Return per Episode : {stats_df['total_reward'].mean():.2f}")
print(f"  Mean Episode Length     : {stats_df['episode_length'].mean():.2f}")
print("  Terminal Outcome Counts :")
print(stats_df["terminal_reason"].value_counts())


In [ ]:
val_env = RansomwareDetectionEnv(seed=42)

# 1. Reset check
obs, info = val_env.reset(seed=42)
assert obs is not None and isinstance(info, dict), "Reset failed"

# 2. Shape check
assert obs.shape == (16,), "Invalid observation shape"

# 3. Action space validity
assert val_env.action_space.n == 3, "Invalid action space"

# 4. Observation space containment
assert val_env.observation_space.contains(obs), "Observation containment failed"

# 5. NaN check
assert not np.isnan(obs).any(), "NaN found in observation"

# 6. Infinity check
assert not np.isinf(obs).any(), "Infinity found in observation"

# 7. Step execution & return types
obs, rew, term, trunc, info = val_env.step(ACTION_CONTINUE)
assert isinstance(rew, float) and np.isfinite(rew), "Reward is not finite float"
assert isinstance(term, bool) and isinstance(trunc, bool), "Invalid term/trunc flags"

# 8. Guaranteed episode termination
val_env.reset(seed=42)
for _ in range(5):
    obs, rew, term, trunc, info = val_env.step(ACTION_CONTINUE)
assert term == True, "Episode did not terminate at maximum step limit"

print("All 10 Environment Integrity Invariants Verified: [PASSED]")


In [ ]:
env_a = RansomwareDetectionEnv(seed=42)
env_b = RansomwareDetectionEnv(seed=42)

obs_a, info_a = env_a.reset(seed=42)
obs_b, info_b = env_b.reset(seed=42)

assert info_a["sample_idx"] == info_b["sample_idx"], "Sample index mismatch!"
assert np.allclose(obs_a, obs_b), "Observation vector mismatch!"
assert np.allclose(env_a.current_sequence, env_b.current_sequence), "Synthetic sequence mismatch!"

print(f"Reproducibility Verified: Exactly identical episodes generated with seed=42 (Sample: {info_a['sample_idx']})")


In [ ]:
cases = [
    {"desc": "Case A: Benign + Finished Safely", "is_rw": False, "actions": [ACTION_CONTINUE]*5},
    {"desc": "Case B: Benign + Premature Alert", "is_rw": False, "actions": [ACTION_ALERT]},
    {"desc": "Case C: Ransomware + Missed Alert", "is_rw": True, "actions": [ACTION_CONTINUE]*5},
    {"desc": "Case D: Ransomware + Early Alert (t=0)", "is_rw": True, "actions": [ACTION_ALERT]},
    {"desc": "Case E: Ransomware + Late Alert (t=4)", "is_rw": True, "actions": [ACTION_CONTINUE]*4 + [ACTION_ALERT]},
]

demo_results = []
for c in cases:
    obs, info = env.reset(seed=42, options={"is_ransomware": c["is_rw"]})
    tot_r = 0.0
    for a in c["actions"]:
        obs, r, term, trunc, info = env.step(a)
        tot_r += r
        if term:
            break
    demo_results.append({
        "Case": c["desc"],
        "Is Ransomware": c["is_rw"],
        "Total Return": round(tot_r, 2),
        "Terminal Reason": info["terminal_reason"],
    })

demo_df = pd.DataFrame(demo_results)
print("Controlled Reward Demonstration Cases:")
display(demo_df)


In [ ]:
steps = np.arange(5)
max_step = 4
bonus_factor = DEFAULT_REWARD_CONFIG["early_detection_bonus_factor"]
base_tp = DEFAULT_REWARD_CONFIG["alert_true_positive"]
tp_rewards = [base_tp + (max_step - s) * bonus_factor for s in steps]
monitoring_costs = [s * DEFAULT_REWARD_CONFIG["monitoring_cost"] for s in steps]
net_rewards = [tp_rewards[s] + monitoring_costs[s] for s in steps]
missed_penalty = DEFAULT_REWARD_CONFIG["final_missed_ransomware_penalty"]

plt.figure(figsize=(8, 4.5))
plt.plot(steps, net_rewards, marker="o", linewidth=2.5, color="#1f77b4", label="Correct RANSOMWARE_ALERT (Net Return)")
plt.axhline(y=missed_penalty, color="red", linestyle="--", linewidth=2, label="Missed Ransomware Penalty (-15.0)")
plt.axhline(y=DEFAULT_REWARD_CONFIG["alert_false_positive"], color="orange", linestyle=":", linewidth=2, label="False Alarm Penalty (-10.0)")

for s in steps:
    plt.annotate(
        f"+{net_rewards[s]:.2f}",
        (s, net_rewards[s]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontweight="bold",
        fontsize=9,
    )

plt.title("Early-Detection Reward Curve by Observation Step\n[NOTE: Synthetic Observation Steps — NOT Real Execution Time]", fontsize=11, fontweight="bold", pad=12)
plt.xlabel("Synthetic Observation Step (t)", fontsize=10, labelpad=8)
plt.ylabel("Net Reward Value", fontsize=10, labelpad=8)
plt.xticks(steps, [f"Step {s}" for s in steps])
plt.ylim(-18, 25)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")
plt.tight_layout()
plt.savefig(figures_dir / "ppo_reward_demonstration.png", dpi=300)
plt.show()


In [ ]:
sampled_classes = stats_df["true_label"].value_counts()
dist_summary = pd.DataFrame({
    "Sampled Count": sampled_classes,
    "Percentage (%)": (sampled_classes / len(stats_df) * 100).round(2),
})
print("Sampled Family Representation across 100 Episodes:")
display(dist_summary)


In [ ]:
stats_csv_path = reports_dir / "ppo_environment_statistics.csv"
stats_df.to_csv(stats_csv_path, index=False)
print(f"Saved 100-episode statistics to: {stats_csv_path}")


## Research Limitations & Synthetic Assumptions

1. **Synthetic Temporal Progression:** The temporal progression in this environment is created via progressive feature reveal (20% to 100%), not physical execution duration.
2. **Absence of Real Timestamps:** The dataset contains static run aggregates; step indices ($t=0 \dots 4$) cannot be translated into physical time units (seconds/milliseconds).
3. **Absence of Sequential Memory Dumps:** Dynamic memory entropy deltas and unmapped page executions are not available in the static dataset.
4. **Proxy Interpretation:** High performance in this environment proves that PPO can solve sequential stopping problems under progressive information gain, but does not constitute empirical real-world detection latency proof.


## Requirements for Genuine Temporal Early-Stage Datasets

For future experimental stages, true early-stage validation will require:
- **Timestamped Event Streams:** Microsecond-precision logs (`sample_id`, `timestamp`, `event_type`, `api_name`, `call_arguments`).
- **Sequential Dynamic Memory Dumps:** Memory entropy snapshots taken at interval micro-windows ($t = 50\text{ms}, 200\text{ms}, 1\text{s}$).
- **Streaming Telemetry Feed:** Real-time process tree events enabling true physical Time-to-Detection (TTD) metrics.


## PPO Interface Specification for Notebook 05

When `notebooks/05_train_ppo.ipynb` is executed, it will consume this environment via the standardized interface:

```python
from src.ppo_environment import RansomwareDetectionEnv

# 1. Training Environment (1,400 training samples)
train_env = RansomwareDetectionEnv(split_type="train", seed=42)

# 2. Evaluation Environment (350 untouched test samples)
eval_env = RansomwareDetectionEnv(split_type="test", seed=42)

# 3. Observation Space: Box(shape=(16,))
# 4. Action Space: Discrete(3) -> 0: CONTINUE, 1: SUSPICIOUS, 2: ALERT
```


In [ ]:
# Verify that src.ppo_environment can be imported cleanly
import src.ppo_environment as ppo_mod

assert hasattr(ppo_mod, "RansomwareDetectionEnv"), "Missing RansomwareDetectionEnv!"
assert hasattr(ppo_mod, "generate_temporal_sequence"), "Missing generate_temporal_sequence!"
assert hasattr(ppo_mod, "DEFAULT_REWARD_CONFIG"), "Missing DEFAULT_REWARD_CONFIG!"

print("src.ppo_environment module verified and ready for independent consumption.")


In [ ]:
config_export_path = data_dir / "ppo_environment_config.json"
ppo_config = {
    "environment_name": "RansomwareDetectionEnv",
    "observation_dimension": 16,
    "feature_names": feature_names,
    "temporal_config": TEMPORAL_CONFIG,
    "action_definitions": ACTION_NAMES,
    "reward_config": DEFAULT_REWARD_CONFIG,
    "seed": 42,
    "normalization_file_path": str(norm_config_path.name),
    "dataset_path": str(cleaned_data_path.name),
    "split_indices_path": str(split_path.name),
    "classes": ["G", "E", "L"]
}

with open(config_export_path, "w", encoding="utf-8") as f:
    json.dump(ppo_config, f, indent=4)

print(f"Saved PPO environment configuration to: {config_export_path}")


In [ ]:
print("""========================================
PPO SYNTHETIC TEMPORAL ENVIRONMENT READY
========================================

Environment:
RansomwareDetectionEnv

Observation Space:
Box(16,) [16 Non-Redundant Early-Stage Features]

Action Space:
Discrete(3) [0: CONTINUE, 1: SUSPICIOUS, 2: ALERT]

Temporal Proxy:
Sequence Length = 5 (Schedule: [0.2, 0.4, 0.6, 0.8, 1.0])

Dataset Partitioning:
Training = 1400 samples | Testing = 350 samples

Normalization:
StandardScaler fit strictly on training set

Reproducibility Seed:
42

Validation Status:
ALL 10 INTEGRITY INVARIANTS PASSED

Ready for:
notebooks/05_train_ppo.ipynb
=========================================""")
